<a href="https://colab.research.google.com/github/ahmedkhanpakistani5-hash/-intern-feedback-to-understand-positive-and-negative-sentiments./blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedkhanpakistani5-hash/ahmad-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse",
                  "fact_content_daily_performance",
                  streaming=True, split="train")

first = next(iter(ds))
print(list(first.keys()))
first

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [ ]:
import pandas as pd
from itertools import islice

df = pd.DataFrame(list(islice(ds, 100_000)))
print(df.shape)
df.head()

(100000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(df["report_date"].min(), df["report_date"].max(), df["report_date"].nunique())

2025-01-27 2025-03-21 41


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:**
Rank pages by how many clicks a fix could win back. A page ranks higher when it gets many search impressions but its click-through rate (CTR) is lower than is normal for the position it ranks at. These pages are being seen but not clicked, so a better title and description are most likely to help. Pages with too few impressions are ignored because their CTR is too noisy. All inputs come from the 21 days before the as-of date only.

**Reason codes (one per page, first match wins):**
- `LOW_DATA` → NO_ACTION: fewer than 100 impressions in the window
- `CTR_BELOW_POSITION` → FIX_TITLE_META: CTR is well below normal for its position
- `HIGH_VOLUME_OK_CTR` → MONITOR: many impressions and CTR is normal, so no fix is needed

In [ ]:
import pandas as pd
import numpy as np

# ---------- setup: clean data, before/after windows ----------
needed = ["report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
          "gsc_impressions", "gsc_clicks", "gsc_sum_position"]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"df is missing columns: {missing}. Rerun your data-load cell."

base = df[df["gsc_data_available"].astype(bool)].copy()
base["report_date"] = pd.to_datetime(base["report_date"])
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

WINDOW = 21
MIN_IMPR = 100
asof = base["report_date"].max() - pd.Timedelta(days=WINDOW)

key = ["client_hash_id", "content_hash_id"]

def agg(x):
    g = x.groupby(key).agg(impr=("gsc_impressions", "sum"),
                           clicks=("gsc_clicks", "sum"),
                           sumpos=("gsc_sum_position", "sum")).reset_index()
    g = g[g["impr"] > 0].copy()
    g["ctr"] = g["clicks"] / g["impr"]
    g["pos"] = g["sumpos"] / g["impr"]
    return g

before = base[(base["report_date"] > asof - pd.Timedelta(days=WINDOW)) & (base["report_date"] <= asof)]
after  = base[base["report_date"] > asof]
assert len(before) > 0 and len(after) > 0, "before or after window is empty. Check your date range."

pages = agg(before)
pages = pages.merge(agg(after)[key + ["impr", "clicks", "ctr"]],
                    on=key, how="left", suffixes=("", "_after"))
pages[["impr_after", "clicks_after"]] = pages[["impr_after", "clicks_after"]].fillna(0)

print("as-of date:", asof.date())
print("before:", before["report_date"].min().date(), "to", before["report_date"].max().date())
print("after: ", after["report_date"].min().date(), "to", after["report_date"].max().date())
print("pages:", pages.shape[0], "\n")

POS_BINS = [-np.inf, 3, 5, 10, 20, np.inf]

def safe_qcut(s, q=5):
    # rank first, so ties can never make the bin edges collide
    return pd.qcut(s.rank(method="first"), q, labels=False, duplicates="drop")

# ---------- signal 1: volume (quick-win flag) ----------
pages["vol_bucket"] = safe_qcut(pages["impr"]) + 1
sig1 = (pages.groupby("vol_bucket")
              .agg(n=("impr", "size"),
                   impr_min=("impr", "min"),
                   impr_max=("impr", "max"),
                   avg_clicks_before=("clicks", "mean"),
                   avg_clicks_after=("clicks_after", "mean"))
              .reset_index())
print("SIGNAL 1: volume (bucket 1 = fewest impressions, 5 = most)")
print(sig1.round(3).to_string(index=False), "\n")

# ---------- signal 2: CTR vs position (CTR-fix flag) ----------
p = pages[(pages["impr"] >= MIN_IMPR) & (pages["impr_after"] >= MIN_IMPR)].copy()
p["pos_band"] = pd.cut(p["pos"], POS_BINS)

band = p.groupby("pos_band", observed=True).agg(c=("clicks", "sum"), i=("impr", "sum"))
normal_ctr = band["c"] / band["i"]              # pooled CTR per band
p["expected_ctr"] = p["pos_band"].map(normal_ctr).astype(float)
p = p[p["expected_ctr"] > 0].copy()

print("SIGNAL 2: CTR vs position")
print("normal CTR per position band:")
print(normal_ctr.round(4).to_string(), "\n")

if len(p) >= 50:
    p["ctr_gap"] = p["ctr"] / p["expected_ctr"]
    p["ctr_gap_after"] = p["ctr_after"] / p["expected_ctr"]
    p["gap_bucket"] = safe_qcut(p["ctr_gap"]) + 1
    sig2 = (p.groupby("gap_bucket")
              .agg(n=("ctr_gap", "size"),
                   gap_before=("ctr_gap", "median"),
                   gap_after=("ctr_gap_after", "median"),
                   mean_gap_after=("ctr_gap_after", "mean"))
              .reset_index())
    print("gap bucket 1 = CTR furthest below normal, 5 = furthest above")
    print(sig2.round(3).to_string(index=False), "\n")
else:
    print("Not enough pages for signal 2:", len(p), "\n")

# ---------- the rule: score, reason code, action ----------
GAP_CUTOFF = 0.5    # "well below normal" = under half the normal CTR
HIGH_VOL_Q = 0.80   # top 20% of impressions = "high volume"

pages["pos_band"] = pd.cut(pages["pos"], POS_BINS)
band_all = (pages[pages["impr"] >= MIN_IMPR]
            .groupby("pos_band", observed=True)
            .agg(c=("clicks", "sum"), i=("impr", "sum")))
expected = band_all["c"] / band_all["i"]

pages["expected_ctr"] = pages["pos_band"].map(expected).astype(float)
pages.loc[pages["expected_ctr"] <= 0, "expected_ctr"] = np.nan
pages["ctr_ratio"] = pages["ctr"] / pages["expected_ctr"]
high_vol = pages["impr"].quantile(HIGH_VOL_Q)

# score = clicks the page is missing compared with a normal page at its position
pages["score"] = (pages["impr"] * (pages["expected_ctr"] - pages["ctr"]).clip(lower=0)).fillna(0)

conditions = [
    pages["impr"] < MIN_IMPR,
    pages["ctr_ratio"].fillna(np.inf) < GAP_CUTOFF,
    pages["impr"] >= high_vol,
]
codes = ["LOW_DATA", "CTR_BELOW_POSITION", "HIGH_VOLUME_OK_CTR"]
pages["reason_code"] = np.select(conditions, codes, default="OK")   # first match wins

action_map = {"LOW_DATA": "NO_ACTION", "CTR_BELOW_POSITION": "FIX_TITLE_META",
              "HIGH_VOLUME_OK_CTR": "MONITOR", "OK": "NO_ACTION"}
pages["action"] = pages["reason_code"].map(action_map)
pages.loc[pages["reason_code"] != "CTR_BELOW_POSITION", "score"] = 0

print("REASON CODES")
print(pages["reason_code"].value_counts().to_string())

as-of date: 2025-02-28
before: 2025-02-08 to 2025-02-28
after:  2025-03-01 to 2025-03-21
pages: 5596 

SIGNAL 1: volume (bucket 1 = fewest impressions, 5 = most)
 vol_bucket    n  impr_min  impr_max  avg_clicks_before  avg_clicks_after
          1 1120         1        14              0.037             0.017
          2 1119        14        62              0.181             0.078
          3 1119        62       147              0.666             0.241
          4 1119       147       312              1.377             0.597
          5 1119       312      6479              4.360             1.746 

SIGNAL 2: CTR vs position
normal CTR per position band:
pos_band
(-inf, 3.0]     0.0151
(3.0, 5.0]      0.0138
(5.0, 10.0]     0.0123
(10.0, 20.0]    0.0081
(20.0, inf]     0.0020 

gap bucket 1 = CTR furthest below normal, 5 = furthest above
 gap_bucket   n  gap_before  gap_after  mean_gap_after
          1 239       0.000      0.000           0.272
          2 239       0.215      0.000 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue (writes the CSV)

The queue ranks every page by the rule's score: the number of clicks a page is missing compared with a normal page at its position. Pages flagged `CTR_BELOW_POSITION` get a score above 0 and rank first. Every other page has score 0 and sits below them, ordered by impressions as a tie-break. Each row has the score, the one reason code, and the action label.

The queue is written to `work/outputs/baseline_action_score.csv` from this notebook. The file is not committed to git by design, because the notebook regenerates it on every run.

In [ ]:
import os
import json
import pandas as pd

# ---------- check that section 1 ran ----------
required = ["client_hash_id", "content_hash_id", "score", "reason_code", "action",
            "impr", "clicks", "ctr", "expected_ctr", "pos"]
assert "pages" in globals(), "pages is not defined. Run the section 1 cell first."
missing = [c for c in required if c not in pages.columns]
assert not missing, f"pages is missing columns: {missing}. Rerun the section 1 cell."

# ---------- build the ranked queue ----------
# sort by score, then by impressions as a tie-break so the order is stable
queue = (pages.sort_values(["score", "impr"], ascending=[False, False])
              [required]
              .reset_index(drop=True))
queue["rank"] = queue.index + 1
queue = queue[["rank"] + required]

# ---------- write the CSV ----------
out_dir = "work/outputs"
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, "baseline_action_score.csv")
queue.to_csv(csv_path, index=False)

# ---------- verify the file was written ----------
check = pd.read_csv(csv_path)
assert len(check) == len(queue), "CSV row count does not match the queue"
assert check["rank"].is_unique, "ranks are not unique"
print("written:", os.path.abspath(csv_path))
print("rows:", len(check))

print("\npages by reason code:")
print(queue["reason_code"].value_counts().to_string())
print("\nflagged for a fix (score > 0):", int((queue["score"] > 0).sum()))

# ---------- small metrics receipt (worth committing, unlike the CSV) ----------
metrics = {
    "as_of_date": str(asof.date()),
    "window_days": int(WINDOW),
    "min_impressions": int(MIN_IMPR),
    "pages_total": int(len(queue)),
    "pages_flagged": int((queue["score"] > 0).sum()),
    "reason_code_counts": {k: int(v) for k, v in queue["reason_code"].value_counts().items()},
}
with open(os.path.join(out_dir, "w04_baseline_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("\nmetrics:", metrics)

# top of the queue, used in section 3
queue.head(10)


written: /content/work/outputs/baseline_action_score.csv
rows: 5596

pages by reason code:
reason_code
LOW_DATA              2814
CTR_BELOW_POSITION    1468
OK                     705
HIGH_VOLUME_OK_CTR     609

flagged for a fix (score > 0): 1468

metrics: {'as_of_date': '2025-02-28', 'window_days': 21, 'min_impressions': 100, 'pages_total': 5596, 'pages_flagged': 1468, 'reason_code_counts': {'LOW_DATA': 2814, 'CTR_BELOW_POSITION': 1468, 'OK': 705, 'HIGH_VOLUME_OK_CTR': 609}}


,rank,client_hash_id,content_hash_id,score,reason_code,action,impr,clicks,ctr,expected_ctr,pos
0,1,client_73cda7b4e4f265ea,content_e352ca3d64eafc24,50.988708,CTR_BELOW_POSITION,FIX_TITLE_META,4432,3,0.000677,0.012182,9.140569
1,2,client_9958f0a7ae1df715,content_d02be57d816cf3d7,35.179477,CTR_BELOW_POSITION,FIX_TITLE_META,3515,16,0.004552,0.014560,4.527738
2,3,client_73cda7b4e4f265ea,content_91d8af19d84c05a6,26.443529,CTR_BELOW_POSITION,FIX_TITLE_META,3770,2,0.000531,0.007545,11.069231
3,4,client_9958f0a7ae1df715,content_00d12fab87754f25,24.129462,CTR_BELOW_POSITION,FIX_TITLE_META,2145,2,0.000932,0.012182,9.570629
4,5,client_9958f0a7ae1df715,content_6cd0c162158858c3,23.884675,CTR_BELOW_POSITION,FIX_TITLE_META,3110,14,0.004502,0.012182,6.633762
5,6,client_73cda7b4e4f265ea,content_5b0a99cac33c373e,19.271240,CTR_BELOW_POSITION,FIX_TITLE_META,1582,0,0.000000,0.012182,9.145386
6,7,client_73cda7b4e4f265ea,content_d5a48fb1b06f4f51,18.567999,CTR_BELOW_POSITION,FIX_TITLE_META,2181,8,0.003668,0.012182,7.416781
7,8,client_73cda7b4e4f265ea,content_722bb352669f7cc0,15.306673,CTR_BELOW_POSITION,FIX_TITLE_META,1667,5,0.002999,0.012182,8.836833
8,9,client_73cda7b4e4f265ea,content_e032cc1f6c7fef57,14.432890,CTR_BELOW_POSITION,FIX_TITLE_META,2252,13,0.005773,0.012182,7.062167
9,10,client_9958f0a7ae1df715,content_14de8e3538fafb02,13.968690,CTR_BELOW_POSITION,FIX_TITLE_META,1984,1,0.000504,0.007545,17.851815


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

For each of the top twenty pages: the action, why it is there, and what would make it wrong.
The "why" comes from the page's numbers. The "wrong if" lines started as drafts built from
each page's own daily data (one-day spikes, shaky position, few active days, noisy CTR), and
I edited the ones I disagreed with.

1. (paste the ten to twenty reviewed lines here, one per row)

In [ ]:
import numpy as np
import pandas as pd

assert "queue" in globals() and "before" in globals(), "Run sections 1 and 2 first."

TOP_N = 20
top = queue.head(TOP_N).copy()

n_flagged = int((queue["score"] > 0).sum())
if n_flagged < TOP_N:
    print(f"WARNING: only {n_flagged} pages are flagged, so the last {TOP_N - n_flagged} rows "
          f"of the top {TOP_N} have score 0. Say this in your write-up.\n")

# ---------- daily detail from the BEFORE window only ----------
d = before.merge(top[key], on=key, how="inner").copy()
d["day_pos"] = np.where(d["gsc_impressions"] > 0,
                        d["gsc_sum_position"] / d["gsc_impressions"].replace(0, np.nan),
                        np.nan)
detail = (d.groupby(key)
            .agg(days_active=("report_date", "nunique"),
                 max_day_impr=("gsc_impressions", "max"),
                 pos_min=("day_pos", "min"),
                 pos_max=("day_pos", "max"))
            .reset_index())
top = top.merge(detail, on=key, how="left")
top["top_day_share"] = (top["max_day_impr"] / top["impr"]).fillna(0)
top["pos_swing"] = (top["pos_max"] - top["pos_min"]).fillna(0)
top["days_active"] = top["days_active"].fillna(0)

# ---------- draft "what would make it wrong" from the page's own data ----------
def wrong_if(r):
    notes = []
    if r["top_day_share"] >= 0.30:
        notes.append(f"{r['top_day_share']:.0%} of its impressions came on a single day, so this may be a one-off spike")
    if r["pos_swing"] >= 5:
        notes.append(f"its daily position swung by {r['pos_swing']:.1f} places, so the average position is unreliable")
    if r["days_active"] <= 7:
        notes.append(f"it was only active on {int(r['days_active'])} days")
    if r["impr"] < 2 * MIN_IMPR:
        notes.append("its impression count is close to the minimum, so the CTR is noisy")
    if r["pos"] <= 3:
        notes.append("it ranks in the top 3, where a low CTR often means a brand or navigational query, or an AI answer taking the click")
    if not notes:
        notes.append("its CTR may recover by itself next window (regression to the mean), so a title fix would get credit it did not earn")
    return "; ".join(notes)

def why(r):
    if r["score"] > 0:
        return (f"{int(r['impr']):,} impressions, CTR {r['ctr']:.2%} vs normal {r['expected_ctr']:.2%} "
                f"at average position {r['pos']:.1f}, about {r['score']:.0f} clicks missing")
    return f"{int(r['impr']):,} impressions, CTR {r['ctr']:.2%}, score 0 ({r['reason_code']})"

print("DRAFT REVIEW LINES (read each one, edit it, then paste into the markdown cell):\n")
for _, r in top.iterrows():
    print(f"{int(r['rank'])}. **{r['action']}** ({r['reason_code']}): {why(r)}.")
    print(f"   *Wrong if:* {wrong_if(r)}.\n")

top[["rank", "action", "reason_code", "impr", "ctr", "expected_ctr", "pos",
     "days_active", "top_day_share", "pos_swing", "score"]].round(3)


DRAFT REVIEW LINES (read each one, edit it, then paste into the markdown cell):

1. **FIX_TITLE_META** (CTR_BELOW_POSITION): 4,432 impressions, CTR 0.07% vs normal 1.22% at average position 9.1, about 51 clicks missing.
   *Wrong if:* its CTR may recover by itself next window (regression to the mean), so a title fix would get credit it did not earn.

2. **FIX_TITLE_META** (CTR_BELOW_POSITION): 3,515 impressions, CTR 0.46% vs normal 1.46% at average position 4.5, about 35 clicks missing.
   *Wrong if:* its CTR may recover by itself next window (regression to the mean), so a title fix would get credit it did not earn.

3. **FIX_TITLE_META** (CTR_BELOW_POSITION): 3,770 impressions, CTR 0.05% vs normal 0.75% at average position 11.1, about 26 clicks missing.
   *Wrong if:* its daily position swung by 5.2 places, so the average position is unreliable.

4. **FIX_TITLE_META** (CTR_BELOW_POSITION): 2,145 impressions, CTR 0.09% vs normal 1.22% at average position 9.6, about 24 clicks missing.
 

,rank,action,reason_code,impr,ctr,expected_ctr,pos,days_active,top_day_share,pos_swing,score
0,1,FIX_TITLE_META,CTR_BELOW_POSITION,4432,0.001,0.012,9.141,17,0.074,2.207,50.989
1,2,FIX_TITLE_META,CTR_BELOW_POSITION,3515,0.005,0.015,4.528,21,0.088,4.150,35.179
2,3,FIX_TITLE_META,CTR_BELOW_POSITION,3770,0.001,0.008,11.069,12,0.123,5.179,26.444
3,4,FIX_TITLE_META,CTR_BELOW_POSITION,2145,0.001,0.012,9.571,21,0.067,5.379,24.129
4,5,FIX_TITLE_META,CTR_BELOW_POSITION,3110,0.005,0.012,6.634,21,0.060,3.874,23.885
5,6,FIX_TITLE_META,CTR_BELOW_POSITION,1582,0.000,0.012,9.145,12,0.113,1.504,19.271
6,7,FIX_TITLE_META,CTR_BELOW_POSITION,2181,0.004,0.012,7.417,17,0.110,6.147,18.568
7,8,FIX_TITLE_META,CTR_BELOW_POSITION,1667,0.003,0.012,8.837,18,0.089,5.693,15.307
8,9,FIX_TITLE_META,CTR_BELOW_POSITION,2252,0.006,0.012,7.062,18,0.085,4.884,14.433
9,10,FIX_TITLE_META,CTR_BELOW_POSITION,1984,0.001,0.008,17.852,21,0.077,15.071,13.969


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## 4. Weak picks + leakage check

**Which picks look wrong and why.** I compared the top picks with what happened in the 21 days
AFTER the as-of date. This is a review only. None of these `_after` numbers feed the score.

Weak picks: ___ of ___. What they have in common: ______

**Leakage check.**
- Every score input (impressions, clicks, CTR, position, expected CTR) comes from the 21 days up to the as-of date.
- `_after` columns were used only in the signal checks (section 1) and this review, never in the score.
- No FlyRank product flags or label columns were used. The score is built only from raw impressions, clicks and position.

In [ ]:
import numpy as np
import pandas as pd

assert "queue" in globals() and "pages" in globals(), "Run sections 1 and 2 first."

# ---------- weak picks: compare the top picks with the AFTER window (review only) ----------
review = queue.head(TOP_N).copy()
check = review.merge(pages[key + ["impr_after", "ctr_after"]], on=key, how="left")
check["impr_after"] = check["impr_after"].fillna(0)

check["ctr_recovered"] = (check["ctr_after"] >= 0.8 * check["expected_ctr"]).fillna(False)
check["dried_up"] = check["impr_after"] < 0.3 * check["impr"]
check["low_volume"] = check["impr"] < 2 * MIN_IMPR
check["not_flagged"] = check["score"] <= 0
check["weak"] = check[["ctr_recovered", "dried_up", "low_volume", "not_flagged"]].any(axis=1)

print(f"{int(check['weak'].sum())} of {len(check)} top picks look weak")
print("  CTR recovered on its own:", int(check["ctr_recovered"].sum()))
print("  impressions dried up:    ", int(check["dried_up"].sum()))
print("  low volume (noisy):      ", int(check["low_volume"].sum()))
print("  not flagged (score 0):   ", int(check["not_flagged"].sum()))

# context: how often does this happen across ALL flagged pages?
fl = pages[pages["reason_code"] == "CTR_BELOW_POSITION"].copy()
if len(fl) > 0:
    rec = (fl["ctr_after"] >= 0.8 * fl["expected_ctr"]).fillna(False)
    print(f"\nacross all {len(fl)} flagged pages, CTR recovered to >=80% of normal: {rec.mean():.0%}")

check[["rank", "action", "impr", "ctr", "ctr_after", "expected_ctr", "impr_after",
       "ctr_recovered", "dried_up", "low_volume", "not_flagged"]].round(4)


8 of 20 top picks look weak
  CTR recovered on its own: 0
  impressions dried up:     8
  low volume (noisy):       0
  not flagged (score 0):    0

across all 1468 flagged pages, CTR recovered to >=80% of normal: 12%


,rank,action,impr,ctr,ctr_after,expected_ctr,impr_after,ctr_recovered,dried_up,low_volume,not_flagged
0,1,FIX_TITLE_META,4432,0.0007,0.0016,0.0122,1231.0,False,True,False,False
1,2,FIX_TITLE_META,3515,0.0046,0.0055,0.0146,1989.0,False,False,False,False
2,3,FIX_TITLE_META,3770,0.0005,0.0053,0.0075,562.0,False,True,False,False
3,4,FIX_TITLE_META,2145,0.0009,0.0000,0.0122,533.0,False,True,False,False
4,5,FIX_TITLE_META,3110,0.0045,0.0025,0.0122,1190.0,False,False,False,False
5,6,FIX_TITLE_META,1582,0.0000,0.0018,0.0122,543.0,False,False,False,False
6,7,FIX_TITLE_META,2181,0.0037,0.0017,0.0122,591.0,False,True,False,False
7,8,FIX_TITLE_META,1667,0.0030,0.0000,0.0122,369.0,False,True,False,False
8,9,FIX_TITLE_META,2252,0.0058,0.0027,0.0122,741.0,False,False,False,False
9,10,FIX_TITLE_META,1984,0.0005,0.0000,0.0075,803.0,False,False,False,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.